# Bài 2 (Trang 15-17) - Dữ liệu Titanic
## Câu 7: Biến đổi các thuộc tính (fare, title, sex, embarked, nhóm tuổi, nhóm giá vé)

**Sinh viên:** Trần Đình Võ - **MSSV:** 6451071088

In [1]:
HO_TEN = "Trần Đình Võ"
MSSV = "6451071088"
print(f"Họ tên: {HO_TEN} - MSSV: {MSSV}")

Họ tên: Trần Đình Võ - MSSV: 6451071088


### Import thư viện và đọc dữ liệu

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

df = pd.read_csv("../Dataset/titanic.csv")

### Các bước xử lý của câu 1 → 6 (chuẩn bị dữ liệu)

In [3]:
df["relatives"] = df["sibsp"] + df["parch"]
df["not_alone"] = (df["relatives"] > 0).astype(int)  # 1: đi cùng người thân, 0: đi một mình

# Tách ký tự đầu của cabin (boong tàu) trước khi xóa cabin, thiếu -> 'U' (Unknown)
deck_map = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7, "T": 8, "U": 0}
df["deck"] = df["cabin"].fillna("U").str[0].map(deck_map).fillna(0).astype(int)

# Xóa các thuộc tính không cần thiết
df = df.drop(columns=["body", "cabin", "boat", "home.dest", "ticket"])

# Điền giá trị thiếu cho age: sinh ngẫu nhiên trong [mean - std, mean + std]
mean = df["age"].mean()
std = df["age"].std()
is_null = df["age"].isnull().sum()
rng = np.random.default_rng(42)
rand_age = rng.integers(int(mean - std), int(mean + std), size=is_null)
df.loc[df["age"].isnull(), "age"] = rand_age

common_value = "S"
df["embarked"] = df["embarked"].fillna(common_value)
# fare thiếu 1 giá trị -> điền trung vị
df["fare"] = df["fare"].fillna(df["fare"].median())

### Câu 7: Biến đổi các thuộc tính (fare, title, sex, embarked, nhóm tuổi, nhóm giá vé)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   pclass     1309 non-null   int64  
 1   survived   1309 non-null   int64  
 2   name       1309 non-null   str    
 3   sex        1309 non-null   str    
 4   age        1309 non-null   float64
 5   sibsp      1309 non-null   int64  
 6   parch      1309 non-null   int64  
 7   fare       1309 non-null   float64
 8   embarked   1309 non-null   str    
 9   relatives  1309 non-null   int64  
 10  not_alone  1309 non-null   int64  
 11  deck       1309 non-null   int64  
dtypes: float64(2), int64(7), str(3)
memory usage: 164.8 KB


In [5]:
# Chuyển giá vé về kiểu nguyên
df["fare"] = df["fare"].astype(int)

In [6]:
# Trích xuất danh hiệu (title) từ tên
titles = {"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4, "Rare": 5}
df["title"] = df["name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
df["title"] = df["title"].replace(["Lady", "Countess", "Capt", "Col", "Don", "Dr",
                                   "Major", "Rev", "Sir", "Jonkheer", "Dona"], "Rare")
df["title"] = df["title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
df["title"] = df["title"].map(titles).fillna(0).astype(int)
df = df.drop(columns=["name"])

In [7]:
df["title"].value_counts().sort_index()

title
1    757
2    264
3    198
4     61
5     29
Name: count, dtype: int64

In [8]:
# Giới tính: 0 = nam, 1 = nữ
df["sex"] = df["sex"].map({"male": 0, "female": 1})
# Cổng lên tàu -> số
ports = {"S": 0, "C": 1, "Q": 2}
df["embarked"] = df["embarked"].map(ports)

In [9]:
# Tuổi -> số nguyên và chia thành 8 nhóm
df["age"] = df["age"].astype(int)
df["age_group"] = pd.cut(df["age"], bins=[-1, 11, 18, 22, 27, 33, 40, 66, np.inf],
                         labels=[0, 1, 2, 3, 4, 5, 6, 7]).astype(int)

In [10]:
# Kiểm tra số phần tử từng nhóm tuổi
df["age_group"].value_counts().sort_index()

age_group
0     91
1    138
2    171
3    218
4    234
5    214
6    234
7      9
Name: count, dtype: int64

In [11]:
# Gợi ý ngưỡng chia fare thành 6 nhóm cân bằng bằng qcut
pd.qcut(df["fare"], 6, duplicates="drop").value_counts().sort_index()

fare
(-0.001, 7.0]    360
(7.0, 8.0]        99
(8.0, 14.0]      214
(14.0, 26.0]     243
(26.0, 53.0]     179
(53.0, 512.0]    214
Name: count, dtype: int64

In [12]:
# Giá vé -> 6 nhóm
df["fare_group"] = pd.cut(df["fare"], bins=[-1, 7.91, 14.454, 31, 99, 250, np.inf],
                          labels=[0, 1, 2, 3, 4, 5]).astype(int)

In [13]:
print(df["fare_group"].value_counts().sort_index())
df.head(10)

fare_group
0    360
1    313
2    325
3    227
4     67
5     17
Name: count, dtype: int64


,pclass,survived,sex,age,sibsp,parch,fare,embarked,relatives,not_alone,deck,title,age_group,fare_group
0,1,1,1,29,0,0,211,0,0,0,2,2,4,4
1,1,1,0,0,1,2,151,0,3,1,3,4,0,4
2,1,0,1,2,1,2,151,0,3,1,3,2,0,4
3,1,0,0,30,1,2,151,0,3,1,3,1,4,4
4,1,0,1,25,1,2,151,0,3,1,3,3,3,4
5,1,1,0,48,0,0,26,0,0,0,5,1,6,2
6,1,1,1,63,1,0,77,0,1,1,4,2,6,3
7,1,0,0,39,0,0,0,0,0,0,1,1,5,0
8,1,1,1,53,2,0,51,0,2,1,3,3,6,3
9,1,0,0,71,0,0,49,1,0,0,0,1,7,3


### Nhận xét
- fare chuyển sang kiểu nguyên, title trích từ name (Mr, Miss, Mrs, Master, Rare), sex và embarked được mã hóa thành số.
- age chia 8 nhóm, fare chia 6 nhóm theo các ngưỡng gợi ý, số phần tử giữa các nhóm tương đối cân bằng.